# A1: RAG with bi-encoder retrieval vs cross-encoder re-ranking
Corpus: *The Income-tax Act, 2025* (572-page gazette PDF).

## 1. Setup
Models, chunk size and k values live here and nowhere else, so nothing can drift between the two pipelines. Temperature is 0 so Answer 1 vs Answer 2 differences come from the chunks, not sampling.

In [1]:
# !pip install -q pypdf sentence-transformers faiss-cpu google-genai requests
import os, re, time, bisect, textwrap
import numpy as np, faiss, requests
from pypdf import PdfReader
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

PDF_URL  = "https://www.taxheal.com/wp-content/uploads/2025/08/income-Tax-Act-2025.pdf"
PDF_PATH = "data/income_tax_act_2025.pdf"
BI_MODEL = "BAAI/bge-small-en-v1.5"
CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # BGE query instruction; chunks get none
CHUNK_TOKENS, OVERLAP_TOKENS = 256, 40
K_RETRIEVE, K_CONTEXT = 10, 3
LLM_MODEL = "gemini-3.5-flash-lite"      # needs GEMINI_API_KEY in the environment
RUN_CHUNK_EXPERIMENT = True         # section 10, re-embeds the corpus at 128 and 512 tokens (slow on CPU)

bi = SentenceTransformer(BI_MODEL)
ce = CrossEncoder(CE_MODEL)
tok = AutoTokenizer.from_pretrained(BI_MODEL)
llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
import torch
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

device: CPU


## 2. Load PDF
The gazette repeats a header on every page, and pypdf emits stray Devanagari glyphs from a stamp graphic; both are stripped line by line. Page numbers are 1-based PDF indices, not printed gazette numbers.

In [2]:
if not os.path.exists(PDF_PATH):
    os.makedirs(os.path.dirname(PDF_PATH), exist_ok=True)
    open(PDF_PATH, "wb").write(requests.get(PDF_URL, headers={"User-Agent": "Mozilla/5.0"}).content)
# strip the repeated gazette header and stray Devanagari glyphs
NOISE_RE = re.compile(r"^\s*\d+\s*$|THE GAZETTE OF INDIA|EXTRAORDINARY|\[Part II[—-]|^Sec\. \d+\]")
DEVANAGARI_RE = re.compile(r"[ऀ-ॿ]+") 

def clean_page(text):
    lines = [l for l in text.splitlines() if not NOISE_RE.search(l.strip())]
    return DEVANAGARI_RE.sub("", "\n".join(lines))

def load_pages(path):
    return [clean_page(p.extract_text() or "") for p in PdfReader(path).pages]

pages = load_pages(PDF_PATH)
print(len(pages), "pages,", sum(map(len, pages)), "chars after cleaning")

572 pages, 1587002 chars after cleaning


## 3. Tokenizer and chunking
A fact cut across two chunks is unreachable for any retriever or re-ranker, so chunking is fixed once and shared by both pipelines.

### 3.1 Choices
* **Size in tokens** (BGE's tokenizer), not characters. BGE silently truncates at 512; the cross-encoder's 512 covers query *plus* chunk, so 256 leaves room for both.
* **Recursive splitting** (paragraph, line, sentence, `; `, word, in that order): a cut lands on the coarsest boundary that fits. A fixed window cuts mid-word.
* **Not section-aware:** the Schedules contain numbered lists (`1.`, `2.`) that look like section headers, so a heading-based splitter would mis-fire.
* **Overlap 40 tokens:** the next chunk starts with the last ~40 tokens of the previous one, so a clause cut at a boundary is whole in one of the two. Cost: near-duplicate chunks, which is why `gold_rank` takes the best rank.
* **Metadata:** each chunk keeps its start page. The splitter is hand-written (same algorithm as LangChain's) because it returns character offsets, which makes the page lookup exact.

In [3]:
SEPARATORS = ["\n\n", "\n", ". ", "; ", " "]

def ntok(s):
    return len(tok(s, add_special_tokens=False)["input_ids"])

def atoms(text, a, b, max_tok, seps=SEPARATORS):
    # recursively cut text[a:b] into contiguous spans of <= max_tok tokens, at the coarsest separator that works
    if ntok(text[a:b]) <= max_tok or not seps:
        return [(a, b)]
    out, pos = [], a
    for m in re.finditer(re.escape(seps[0]), text[a:b]):
        out += atoms(text, pos, a + m.end(), max_tok, seps[1:]); pos = a + m.end()
    if pos < b:
        out += atoms(text, pos, b, max_tok, seps[1:])
    return out

def chunk_text(text, size=CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    # greedily merge adjacent spans up to `size` tokens; start the next chunk by re-using trailing spans worth <= `overlap`
    at = [(a, b, ntok(text[a:b])) for a, b in atoms(text, 0, len(text), size)]
    spans, i = [], 0
    while i < len(at):
        j, n = i, 0
        while j < len(at) and n + at[j][2] <= size:
            n += at[j][2]; j += 1
        j = max(j, i + 1)
        spans.append((at[i][0], at[j - 1][1]))
        if j >= len(at):
            break
        k, o = j, 0
        while k > i + 1 and o + at[k - 1][2] <= overlap:
            k -= 1; o += at[k][2]
        i = k if k > i else j
    return spans

def build_chunks(pages, size=CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    full, starts = "", []
    for p in pages:
        starts.append(len(full)); full += p + "\n"
    return [{"text": full[a:b], "page": bisect.bisect_right(starts, a)} for a, b in chunk_text(full, size, overlap)]

chunks = build_chunks(pages)
print(len(chunks), "chunks")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (352123 > 512). Running this sequence through the model will result in indexing errors


1631 chunks


### 3.2 Checks
No chunk may exceed the model limit (the excess would silently go unembedded). Ending in punctuation is only a rough proxy for "not cut mid-clause". Below it, 20 random chunks (head and tail) to read.

In [4]:
lens = np.array([ntok(c["text"]) for c in chunks])
print(f"tokens/chunk: min {lens.min()}  median {int(np.median(lens))}  max {lens.max()}   (BGE limit 512, CE limit 512 incl. query)")
assert lens.max() <= 512

n_mid = sum(not c["text"].rstrip().endswith((".", ";", ":", "—", "–", ")")) for c in chunks)
print(f"{n_mid}/{len(chunks)} chunks ({100*n_mid/len(chunks):.0f}%) do not end on . ; : or a closing bracket\n")

import random
for i in sorted(random.Random(0).sample(range(len(chunks)), 20)):
    t = chunks[i]["text"].replace("\n", " ")
    print(f"[{i} p{chunks[i]['page']}] {t[:100]} … {t[-100:]}\n")

tokens/chunk: min 137  median 250  max 256   (BGE limit 512, CE limit 512 incl. query)
1099/1631 chunks (67%) do not end on . ; : or a closing bracket

[82 p30] programme rec orded on any disc, tape, perforated media or other  information storage device and inc … eration— (i) for any construction, assembly, mining or like project undertaken by the recipient; or 

[194 p70] respect of that block of asset in relation to that tax year. (3) Where any asset, forming part of a  … ds of said company as on the  date of conversion of the company into limited liability partnership. 

[285 p102] of workers employed in that business or, all businesses of that class; (28) “securities transaction  … d (r) boarding and lodging; (30) “small enterprise” shall be an enterprise classified as such under 

[286 p103] (p) processing; (q) supply of electrical or other energy; and (r) boarding and lodging; (30) “small  …  of a forward market or a stock exchange in the course of any transaction in the nature o

**Notes from the printed chunks** 
* Few end on a complete sentence; others end mid-sentence, mid-clause or mid-list.
* None of the 20 starts at the start of a section. Most open mid-sentence; others open at a list item whose lead-in is in the previous chunk.
* pypdf leaves stray spaces inside words. Gold matching ignores whitespace, but the embedder sees the broken words.
* Marginal headings are mixed into the text, sometimes mid-clause
* Tables and formulas are flattened
* Overlap rescued a cut: the k5 sentence is split between chunks 123 (ends "…shall not be included in") and 124, but 124 holds it whole.

## 4. Index
Chunk vectors do not depend on the query, so they are embedded once, offline. Vectors are normalised, so inner product equals cosine. A flat (exact) FAISS index is enough at this size and means any miss is the model's fault, not the index's approximation.

In [5]:
def build_index(chunks, model=bi):
    vecs = model.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(np.asarray(vecs, dtype="float32"))
    return index

t0 = time.perf_counter()
index = build_index(chunks)
print(f"{index.ntotal} vectors, dim {index.d}, built in {time.perf_counter()-t0:.0f}s (offline cost, not counted in query latency)")

1631 vectors, dim 384, built in 88s (offline cost, not counted in query latency)


## 5. Pipeline 1: dense Top 10, Top 3 to the LLM
`dense_top_k` is the shared candidate generator both pipelines start from exactly this list. 
The prompt is one template with the usual parts: instruction, labelled evidence `[S1]`..`[S3]`, a refusal clause, a citation rule, question last. It is used unchanged for both answers.

**Dense vs sparse.** Dense retrieval (used here) compares embedding vectors, so it matches meaning and tolerates paraphrase, but can miss exact terms and numbers. Sparse retrieval (BM25/TF-IDF) matches the query's literal terms: exact on rare words, blind to synonyms.There is no sparse or hybrid run here.

In [6]:
def dense_top_k(query, index=index, k=K_RETRIEVE, model=bi):
    qv = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    _, ids = index.search(np.asarray(qv, dtype="float32"), k)
    return [int(i) for i in ids[0]]

PROMPT = '''You answer questions about the Income-tax Act, 2025 using ONLY the evidence below.
If the evidence does not contain the answer, reply exactly: "Not found in the provided evidence."
Cite the evidence you used as [S1], [S2] or [S3] after each claim.

{evidence}

Question: {question}
Answer:'''

def build_prompt(query, chunk_ids, chunks=chunks):
    ev = "\n\n".join(f"[S{n}] (page {chunks[i]['page']})\n{chunks[i]['text']}" for n, i in enumerate(chunk_ids, 1))
    return PROMPT.format(evidence=ev, question=query)

MIN_GAP_S = 13          # free tier allows 5 requests/min per model; spacing calls avoids 429s
_last_call, LAST_LLM_MS = 0.0, None

def generate(query, chunk_ids, retries=5):
    global _last_call, LAST_LLM_MS
    cfg = types.GenerateContentConfig(temperature=0)
    for attempt in range(retries):
        time.sleep(max(0.0, MIN_GAP_S - (time.time() - _last_call)))
        _last_call = time.time()
        try:
            t0 = time.perf_counter()
            text = llm.models.generate_content(model=LLM_MODEL, contents=build_prompt(query, chunk_ids), config=cfg).text.strip()
            LAST_LLM_MS = (time.perf_counter() - t0) * 1e3      # API call only, throttle sleeps excluded
            return text
        except Exception:
            if attempt == retries - 1:
                raise                      # fail loudly rather than store an error string as an "answer"
            time.sleep(35)

def pipeline1(query):
    cands = dense_top_k(query)
    return cands, cands[:K_CONTEXT]      # baseline: bi-encoder order

## 6. Pipeline 2: cross-encoder re-rank of the same 10
The cross-encoder reads query and chunk together in one forward pass per pair, so it can only run on the shortlist. It sees the **same** 10 candidates; it only reorders them. Scores are raw logits and only comparable within one query.

In [7]:
def rerank(query, candidate_ids, chunks=chunks):
    # one forward pass per (query, chunk) pair; returns ids sorted by descending score
    scores = ce.predict([(query, chunks[i]["text"]) for i in candidate_ids], show_progress_bar=False)
    order = np.argsort(-scores)
    return [candidate_ids[i] for i in order], scores[order]

def pipeline2(query, cands=None):
    cands = cands or dense_top_k(query)
    ranked, _ = rerank(query, cands)
    return ranked, ranked[:K_CONTEXT]

# warm-up so one-off setup cost is not in any later timing
_ = pipeline2("what is the standard deduction for salary income")
q = "If someone lives in their own flat but lets it out for a weekend, is the flat still valued at nil?"
cands, ctx1 = pipeline1(q); ranked, ctx2 = pipeline2(q, cands)
print("baseline order  :", cands); print("re-ranked order :", ranked)
# (no LLM call here: the free Gemini tier is capped at 20 requests per day, so answers are generated only in section 7)

baseline order  : [124, 351, 492, 1555, 300, 128, 461, 134, 373, 365]
re-ranked order : [124, 300, 373, 351, 128, 134, 492, 365, 461, 1555]


## 7. Failure tests
`CANDIDATES` holds 40 queries across the four failure types. Each carries a **gold phrase**: a short string only the gold chunk contains. Queries are paraphrased, not copied from the gold text, so the bi-encoder does not win trivially. The gold chunk's rank is found by substring match (whitespace-insensitive), so it survives any change to chunk size. Where overlap makes two chunks contain the phrase, the better rank is used.

Re-ranking only reorders the Top 10, so a failure is *recoverable* only if the gold chunk is already in it.

In [8]:
CANDIDATES = [
 # (id, type, query, gold phrase)
 ("k1","keyword","How much money can I borrow from a friend in cash before the transaction has to go through a bank?","No person shall take or accept from another person any loan or deposit"),
 ("k2","keyword","Which insurance costs can a shop deduct for protecting its inventory?","insurance against risk of damage or destruction of stocks or stores"),
 ("k3","keyword","What share of an employee's pay can the employer put into the staff pension scheme and still deduct it?","up to 14% of the salary of the employee"),
 ("k4","keyword","From what age is a person treated as an elderly taxpayer?","age of sixty years or more at any time during the relevant tax year"),
 ("k5","keyword","Landlord could not collect the rent from the tenant; is it still counted as his earnings?","The rent which cannot be realised by the owner shall not be included"),
 ("k6","keyword","What percentage cut applies to rent received late for earlier years?","A sum equal to 30% of the arrears of rent"),
 ("k7","keyword","How much of the salary is knocked off automatically for an employee?","₹ 75000 or the salary, whichever is less"),
 ("k8","keyword","How many months must I own a listed share before selling it counts as a long-term sale?","the words “twelve months” had been substituted"),
 ("e1","exception","I live in my own flat but let it out for a weekend. Do I still pay nothing on it?","shall not apply, if the house or any part thereof is actually let"),
 ("e2","exception","I own three homes and live in none of them. How many can be shown as having zero value?","shall apply only in respect of two of such houses"),
 ("e3","exception","My home loan interest deduction is capped at a lower figure when the house is not finished in time. What is the rule?","completed within five years from the end of tax year in which capital was borrowed"),
 ("e4","exception","A business paid a supplier fifteen thousand rupees in notes. Can it claim that cost?","exceeds ₹10000 and is not made through specified banking or online mode"),
 ("e5","exception","Are farmers lending to each other in notes covered by the bank-transfer requirement?","both, have agricultural income and neither has any income chargeable"),
 ("e6","exception","What is the bigger threshold for borrowing from a village credit cooperative?","primary agricultural credit society or a primary"),
 ("e7","exception","When is equipment that was earlier used abroad still considered brand new for the depreciation claim?","shall not be regarded as machinery or plant previously used for any purpose"),
 ("e8","exception","An undertaking was destroyed in a flood and then revived. Which condition is waived for it?","the business of such undertaking is re-established, re-constructed or"),
 ("e9","exception","For an asset received as inheritance, what is the starting cost taken?","Actual cost to the previous owner as reduced by"),
 ("n1","negation","Can a company deduct a provision it set aside for retirement payments to staff?","no deduction shall be allowed for any provision made for the payment of gratuity"),
 ("n2","negation","Is a provision for retirement benefits deductible before it is paid out?","no deduction shall be allowed for any provision made for the payment of gratuity"),
 ("n3","negation","Who is not bound by the ban on taking large amounts in notes?","any receipt by Government, any banking company"),
 ("n4","negation","Are joint owners of a rented building taxed as one group?","co-owners shall not be assessed as an association of persons"),
 ("n5","negation","Can interest paid to a lender abroad be claimed as a cost if no tax was withheld?","shall not be allowed as a deduction under this section, if—"),
 ("n6","negation","Which deductions are not permitted while computing income from capital gains?","securities transaction tax under Chapter VII"),
 ("n7","negation","Is an employer barred from deducting money paid to set up a staff fund that is not recognised?","No deduction shall be allowed in respect of any sum paid by the assessee as an employer towards setting up"),
 ("n8","negation","Is paying for staff health cover in cash a deductible cost?","through any mode of payment other than cash, to effect or to keep in force an insurance on the health of its employees"),
 ("n9","negation","Can a partner of a firm claim relief on interest earned in the firm's savings account?","in computing the total income of any partner of the firm"),
 ("n10","negation","Do factories making tobacco products qualify for the profit-linked tax holiday?","which pertains to tobacco and manufactured tobacco substitutes"),
 ("n11","negation","Does money spent on buying land count towards the research deduction for a company?","not being expenditure in the nature of cost of any land or building"),
 ("b1","buried","Unsold flats held by a builder: how is the value of the property treated?","annual value of such property or part thereof shall be nil for two years"),
 ("b2","buried","If I transfer a house to my wife for nothing, who is taxed on its income?","transfers without adequate consideration, any property to the spouse"),
 ("b3","buried","At what ownership level is someone considered to have a major stake in a business?","carrying not less than 20% of the voting power"),
 ("b4","buried","A bill claimed last year is settled in cash this year above the limit. What happens to it?","shall be deemed to be the income under the head “Profits and gains of business or profession” in such subsequent tax year"),
 ("b5","buried","If the tenant pays for repairs to the rented premises, can the tenant deduct that?","where he has undertaken to bear the cost of repairs to the premises"),
 ("b6","buried","Does every shopkeeper have to accept digital payments?","Every person shall provide facility for accepting payment"),
 ("b7","buried","What exemption exists for soldiers' retirement gratuity?","Payment of retiring gratuity received under the Pension Code"),
 ("b8","buried","Limit on tax-free encashment of unused leave when several employers paid it in the same year?","cash equivalent of leave salary is received from one or more than one employer"),
 ("b9","buried","Which payments to an employee's provident fund can the employer deduct?","recognised provident fund or an approved superannuation fund"),
 ("b10","buried","When an employer pays staff contributions late, are they still deductible?","if it is credited by the assessee to the account of the employee in the relevant fund or funds by the due date"),
 ("b11","buried","Are owners of partly business-use premises allowed to deduct the full rent?","restricted to the fair proportionate part thereof"),
 ("b12","buried","If someone shares a flat under a housing society scheme, are they treated as owner?","a member of a co-operative society, company or other association"),
]
print(len(CANDIDATES), "candidate queries")

40 candidate queries


In [9]:
_norm = lambda s: re.sub(r"\s+", "", s).lower()

def gold_rank(order, phrase, chunks=chunks):
    # 1-based position of the best-placed chunk containing the phrase, or None if absent from `order`
    hits = [pos for pos, i in enumerate(order, 1) if _norm(phrase) in _norm(chunks[i]["text"])]
    return min(hits) if hits else None

rows = []
for qid, typ, q, gold in CANDIDATES:
    cands = dense_top_k(q, k=50)
    base = gold_rank(cands[:K_RETRIEVE], gold)
    ranked, _ = rerank(q, cands[:K_RETRIEVE])
    rows.append(dict(id=qid, type=typ, query=q, gold=gold, base=base, rerank=gold_rank(ranked, gold),
                     base_in_50=gold_rank(cands, gold)))

def status(r):
    b, rr = r["base"], r["rerank"]
    if b is None:  return "gold not in Top 10 (retrieval miss)"
    if b > K_CONTEXT and rr <= K_CONTEXT: return "RECOVERED (rank 4-10 -> Top 3)"
    if b <= K_CONTEXT and rr > K_CONTEXT: return "HURT (Top 3 -> out)"
    if b <= K_CONTEXT and rr <= K_CONTEXT: return "already in Top 3"
    return "still outside Top 3"

print(f"{'id':4}{'type':10}{'base':>5}{'rerank':>7}{'in50':>6}  status")
for r in rows:
    print(f"{r['id']:4}{r['type']:10}{str(r['base']):>5}{str(r['rerank']):>7}{str(r['base_in_50']):>6}  {status(r)}")
from collections import Counter
print(Counter(status(r) for r in rows))
in3 = lambda k: sum(r[k] is not None and r[k] <= K_CONTEXT for r in rows)
print(f"gold in Top 3: baseline {in3('base')}/{len(rows)}, re-ranked {in3('rerank')}/{len(rows)};  in Top 10: {sum(r['base'] is not None for r in rows)}/{len(rows)}")

id  type       base rerank  in50  status
k1  keyword       2      4     2  HURT (Top 3 -> out)
k2  keyword       3      1     3  already in Top 3
k3  keyword       2      3     2  already in Top 3
k4  keyword       3      4     3  HURT (Top 3 -> out)
k5  keyword      10      1    10  RECOVERED (rank 4-10 -> Top 3)
k6  keyword    None   None    11  gold not in Top 10 (retrieval miss)
k7  keyword    None   None  None  gold not in Top 10 (retrieval miss)
k8  keyword       2      2     2  already in Top 3
e1  exception    10      2    10  RECOVERED (rank 4-10 -> Top 3)
e2  exception     1      2     1  already in Top 3
e3  exception    10      2    10  RECOVERED (rank 4-10 -> Top 3)
e4  exception     3      8     3  HURT (Top 3 -> out)
e5  exception  None   None    30  gold not in Top 10 (retrieval miss)
e6  exception     5      7     5  still outside Top 3
e7  exception     2      6     2  HURT (Top 3 -> out)
e8  exception     1      1     1  already in Top 3
e9  exception  None   None   


**Screening result** (40 queries, one run)

| type | n | already Top 3 | recovered | hurt | still outside Top 3 | gold not in Top 10 |
|---|---|---|---|---|---|---|
| keyword | 8 | 3 | 1 | 2 | 0 | 2 |
| exception | 9 | 2 | 2 | 2 | 1 | 2 |
| negation | 11 | 6 | 1 | 1 | 1 | 2 |
| buried | 12 | 7 | 1 | 1 | 0 | 3 |
| all | 40 | 18 | 5 | 6 | 2 | 9 |

* Gold in Top 3: baseline 24/40, re-ranked 23/40. Re-ranking recovered 5 and hurt 6, so on this set it did not improve the Top 3. A one-query difference is within noise.
* 9/40 are retrieval misses the cross-encoder cannot touch: 7 have gold at dense rank 11–50 (k6 is rank 11), 2 (k7, b4) are beyond 50.
* The 6 hurt queries lose gold to ranks 4, 4, 6, 7, 8, 8 (k1, k4, e7, n7, e4, b2).


### 7.1 The kept queries
Picked from the screening table: five failures that re-ranking recovered (k5, e1, e3, n2, b8), one control the baseline gets right (b5), and two limit cases (k6 a retrieval miss, n7 where re-ranking made it worse). The pick was made after seeing the results, so quote the screening-table rates, not the rate among these.

In [10]:
FINAL = {"k5":"Keyword trap", "e1":"Exception", "e3":"Exception", "n2":"Negation", "b8":"Buried chunk", "b5":"Control (baseline correct)",
         "n7":"Limit: re-ranking hurt", "k6":"Limit: retrieval miss"}
byid = {r["id"]: r for r in rows}
results = {}
for qid, label in FINAL.items():
    r = byid[qid]
    cands, ctx1 = pipeline1(r["query"]); ranked, ctx2 = pipeline2(r["query"], cands)
    results[qid] = dict(cands=cands, ranked=ranked, a1=generate(r["query"], ctx1), a2=generate(r["query"], ctx2), ctx1=ctx1, ctx2=ctx2)
    print(f"\n=== {qid} [{label}] ===\n{r['query']}\n gold rank: baseline {r['base']} -> re-ranked {r['rerank']}   ({status(r)})")
    print(f"  baseline top-1 (p{chunks[cands[0]]['page']}): {chunks[cands[0]]['text'][:120]!r}")
    for tag, ids in (("baseline", cands), ("re-ranked", ranked)):
        print(f"  {tag:10}", [f"{i}(p{chunks[i]['page']})" for i in ids[:5]], "...")
    print("  Answer 1:", textwrap.fill(results[qid]["a1"], 110, subsequent_indent="            "))
    print("  Answer 2:", textwrap.fill(results[qid]["a2"], 110, subsequent_indent="            "))


=== k5 [Keyword trap] ===
Landlord could not collect the rent from the tenant; is it still counted as his earnings?
 gold rank: baseline 10 -> re-ranked 1   (RECOVERED (rank 4-10 -> Top 3))
  baseline top-1 (p48): 'which such rent is received or realised.\n(2) The amount deemed to be income from house property under\nsub-section (1) sh'
  baseline   ['128(p48)', '1368(p467)', '135(p50)', '127(p48)', '1001(p341)'] ...
  re-ranked  ['124(p47)', '135(p50)', '128(p48)', '127(p48)', '134(p50)'] ...
  Answer 1: Not found in the provided evidence.
  Answer 2: The rent which cannot be realised by the owner shall not be included in computing the actual rent received or
            receivable, subject to the rules as may be made in this behalf [S1].



=== e1 [Exception] ===
I live in my own flat but let it out for a weekend. Do I still pay nothing on it?
 gold rank: baseline 10 -> re-ranked 2   (RECOVERED (rank 4-10 -> Top 3))
  baseline top-1 (p541): '(b) is withdrawn by the assessee from the deposit account, \nduring any tax year for utilisation for the purposes of such'
  baseline   ['1555(p541)', '134(p50)', '128(p48)', '1478(p506)', '365(p130)'] ...
  re-ranked  ['134(p50)', '124(p47)', '135(p50)', '1478(p506)', '128(p48)'] ...
  Answer 1: Not found in the provided evidence.
  Answer 2: Not found in the provided evidence.



=== e3 [Exception] ===
My home loan interest deduction is capped at a lower figure when the house is not finished in time. What is the rule?
 gold rank: baseline 10 -> re-ranked 2   (RECOVERED (rank 4-10 -> Top 3))
  baseline top-1 (p166): 'registered in India with the main object of carrying on the business of \nproviding long-term finance for construction or'
  baseline   ['480(p166)', '479(p166)', '125(p47)', '481(p166)', '478(p165)'] ...
  re-ranked  ['479(p166)', '126(p47)', '481(p166)', '480(p166)', '478(p165)'] ...
  Answer 1: Not found in the provided evidence.
  Answer 2: Not found in the provided evidence.



=== n2 [Negation] ===
Is a provision for retirement benefits deductible before it is paid out?
 gold rank: baseline 9 -> re-ranked 1   (RECOVERED (rank 4-10 -> Top 3))
  baseline top-1 (p566): 'withdrawal is made or , pension \nis received, and shall be liable to \ntax in the said year. \n \n  \n \n  A B C D \n  (b) as '
  baseline   ['1618(p566)', '469(p163)', '1617(p566)', '1589(p553)', '519(p179)'] ...
  re-ranked  ['138(p51)', '175(p64)', '519(p179)', '1446(p491)', '1618(p566)'] ...
  Answer 1: Not found in the provided evidence.
  Answer 2: Subject to the provisions of sub-section (1)(d), no deduction shall be allowed for any provision made for the
            payment of gratuity to the employees on their retirement or termination for any reason [S1].
            Furthermore, in case deduction has been allowed for any provision made under sub-section (1)(d),
            then no deduction shall be allowed on actual payment made from such provision [S1].



=== b8 [Buried chunk] ===
Limit on tax-free encashment of unused leave when several employers paid it in the same year?
 gold rank: baseline 6 -> re-ranked 1   (RECOVERED (rank 4-10 -> Top 3))
  baseline top-1 (p178): 'employed during the tax year; or\n(ii) emoluments paid or payable to employees employed during the\ntax year, where that y'
  baseline   ['517(p178)', '1578(p550)', '1579(p550)', '220(p78)', '519(p179)'] ...
  re-ranked  ['120(p45)', '117(p44)', '519(p179)', '517(p178)', '101(p37)'] ...
  Answer 1: Not found in the provided evidence.
  Answer 2: Based on the provided evidence, if any payment on account of cash equivalent of leave salary is received from
            one or more than one employer in the same tax year, the aggregate amount of deduction shall not
            exceed $A - B$, where $A$ is the limit specified by the Central Government by notification, and
            $B$ is the aggregate amount of payment or payments which was received in any one or more earli


=== b5 [Control (baseline correct)] ===
If the tenant pays for repairs to the rented premises, can the tenant deduct that?
 gold rank: baseline 1 -> re-ranked 1   (already in Top 3)
  baseline top-1 (p50): '(c) rent paid, when the premises are occupied by the assessee as a tenant;\n(d) amount paid on account of current repairs'
  baseline   ['135(p50)', '134(p50)', '862(p294)', '153(p56)', '128(p48)'] ...
  re-ranked  ['135(p50)', '134(p50)', '127(p48)', '128(p48)', '126(p47)'] ...
  Answer 1: Yes, the amount paid on account of cost of repairs (not being in the nature of capital expenditure) shall be
            allowed as a deduction when the premises are occupied by the assessee as a tenant and where the
            assessee has undertaken to bear the cost of repairs to the premises [S1].
  Answer 2: Yes, the amount paid on account of cost of repairs (not being in the nature of capital expenditure) shall be
            allowed as a deduction when the premises are occupied by the ass


=== n7 [Limit: re-ranking hurt] ===
Is an employer barred from deducting money paid to set up a staff fund that is not recognised?
 gold rank: baseline 1 -> re-ranked 7   (HURT (Top 3 -> out))
  baseline top-1 (p51): 'award, contract of service or otherwise and the provisions of section 37 shall \nnot apply for determining the “due date”'
  baseline   ['138(p51)', '136(p51)', '1584(p552)', '1578(p550)', '1580(p550)'] ...
  re-ranked  ['1584(p552)', '1580(p550)', '136(p51)', '1578(p550)', '1166(p397)'] ...
  Answer 1: No deduction shall be allowed in respect of any sum paid by the assessee as an employer towards setting up or
            formation of, or as contribution to, any fund, trust, company, association of persons, body of
            individuals, society registered under the Societies Registration Act, 1860, or other institution
            for any purpose, except where such sum is so paid for the purposes and to the extent provided by
            or under sub-section (1)(a) o


=== k6 [Limit: retrieval miss] ===
What percentage cut applies to rent received late for earlier years?
 gold rank: baseline None -> re-ranked None   (gold not in Top 10 (retrieval miss))
  baseline top-1 (p448): 'if the advance tax paid by the assessee on the current income,––\n(a) on or before the 15th day of June is 12% or more of'
  baseline   ['1304(p448)', '641(p221)', '363(p130)', '365(p130)', '351(p126)'] ...
  re-ranked  ['126(p47)', '1304(p448)', '1303(p447)', '363(p130)', '186(p67)'] ...
  Answer 1: Not found in the provided evidence.
  Answer 2: Not found in the provided evidence.


### 7.2 Verdicts and failing stage
Checked against the gold chunk text printed below the table. "Refused" means the reply was "Not found in the provided evidence.";

| id | type | gold rank (base → rerank) | gold chunk, page | Answer 1 | Answer 2 | failing stage |
|---|---|---|---|---|---|---|
| k5 | keyword | 10 → 1 | 123–124, p47 | refused | correct | baseline ranking; top-1 was chunk 128 (rent *received later*, same words "rent", "realised") |
| e1 | exception | 10 → 2 | 124–125, p47 | refused | refused, though S2 (chunk 124) holds "(7)(b) shall not apply, if … actually let" | A1 ranking; A2 generation |
| e3 | exception | 10 → 2 | 126, p47 | refused | refused, though S2 (chunk 126) holds ₹200000 and ₹30000 "in any other case" | A1 ranking; A2 generation |
| n2 | negation | 9 → 1 | 138, p51 | refused | correct (section 29(2), gratuity only; does not say what (1)(d) allows) | baseline ranking |
| b8 | buried | 6 → 1 | 120, p45 | refused | correct (the A − B formula) | baseline ranking |
| b5 | control | 1 → 1 | 135, p50 | correct | correct (identical) | none |
| n7 | negation | 1 → 7 | 138, p51 | correct | refused | re-ranking |
| k6 | keyword | 11 → miss | 128, p48 | refused | refused | retrieval: gold at dense rank 11, just outside K=10 |

* No answer here is a hallucination or a misstated rule; every failure is a refusal.
* e1 and e3 count as "recovered" by rank but not by answer: the gold chunk was in context and the model still refused.
* e1's Answer 2 was a correct answer in an earlier run of this notebook (same chunks, prompt, temperature 0), so LLM output is not stable between runs.


In [11]:
# gold chunk(s) for each kept query, to check the answers against
def show_gold(qid):
    for i, c in enumerate(chunks):
        if _norm(byid[qid]["gold"]) in _norm(c["text"]):
            print(f"[{qid}] gold chunk {i}, page {c['page']}\n{c['text']}\n")

for qid in FINAL: show_gold(qid)

[k5] gold chunk 123, page 47
which are chargeable to income-tax.
21.( 1) For the purposes of section 20, the annual value of any property shall
be deemed to be the higher of the following:—
(a) the sum for which it might reasonably be expected to let from year
to year; or
(b) the actual rent received or receivable by the owner, if the property
or any part of it is let.
(2)I f the property or any part of it islet and was vacant for the whole or any
part of the tax year and owing to such vacancy the actual rent received or 
receivable by th e owner in respect thereof is less than the sum referred to in       
sub-section (1)(a), the annual value of such property shall be deemed to be the 
amount so received or receivable.
(3) The annual value of the property shall be reduced by the taxes (including
service taxes) levied by a local authority in respect of such property, actually paid 
during the tax year by the owner, irrespective of when such taxes became payable.
(4) The rent which cann

[e3] gold chunk 126, page 47
(a) ₹ 200000, subject to the following conditions:––
(i) the property has been acquired or constructed with borrowed
capital and such acquisition or construction is completed within five 
years from the end of tax year in which capital was borrowed;
(ii) the assessee furnishes a certificate from the person to whom
interest is payable on such capital; and
Determination 
of annual 
value.
Deductions from 
income from 
house property.
Arrears of rent 
and unrealised 
rent received 
subsequently.
Property owned 
by co-owners.
Interpretation.
(b) ₹30000 in any other case.
(3) The deduction under section 22( 1)(c) shall be computed after
reducing the interest referred to in the said section by any amount already 
allowed as a deduction under any other provisions of this Act.
(4) The certificate referred to in sub-section (2) shall specify––
(a) the amount of interest payable on capital borrowed; and
(b) the interest payable on any new loan, where subsequent to th

[b5] gold chunk 135, page 50
(c) rent paid, when the premises are occupied by the assessee as a tenant;
(d) amount paid on account of current repairsto the premises, not being in
the nature of capital expenditure, when the premises are occupied by the assessee 
otherwise than as a tenant;
(e) amount paid on account of cost of repairs, not being in the nature of
capital expenditure, when the premises are occupied by the assessee as a tenant 
and where he has undertaken to bear the cost of repairs to the premises; and
(f) the amount paid on account of current repairs to machinery, plant or
furniture, not being in the nature of capital expenditure.
(2) In case where the premises, building, machinery, plant or furniture is partly
used or not wholly and exclusively used for the purposes of the business or profession, 
the deduction allowable under sub -section ( 1) shall be restricted to the fair 
proportionate part thereof as determined by the Assessing Officer, having regard to 
the usage

## 8. Latency
Models are loaded and warmed up. Each stage is timed with `perf_counter`, 20 repeats per kept query, on the device printed in section 1. LLM time is reported separately (network-bound, noisy) from only 3 calls.

In [12]:
REPS = 20
timing_queries = [byid[q]["query"] for q in FINAL]
t_embed, t_search, t_ce = [], [], []
for q in timing_queries:
    for _ in range(REPS):
        t0 = time.perf_counter(); qv = bi.encode([QUERY_PREFIX + q], normalize_embeddings=True); t1 = time.perf_counter()
        _, ids = index.search(np.asarray(qv, dtype="float32"), K_RETRIEVE); t2 = time.perf_counter()
        cands = [int(i) for i in ids[0]]
        ce.predict([(q, chunks[i]["text"]) for i in cands], show_progress_bar=False); t3 = time.perf_counter()
        t_embed.append((t1-t0)*1e3); t_search.append((t2-t1)*1e3); t_ce.append((t3-t2)*1e3)

t_llm = []
for q in timing_queries[:3]:
    ctx = results[[k for k in FINAL if byid[k]["query"] == q][0]]["ctx1"]
    generate(q, ctx); t_llm.append(LAST_LLM_MS)

ms = lambda x: f"{np.mean(x):8.1f} ± {np.std(x):6.1f}"
tot1 = np.array(t_embed) + np.array(t_search); tot2 = tot1 + np.array(t_ce)
print(f"{'stage (ms/query, mean ± std)':36}{'Pipeline 1':>20}{'Pipeline 2':>20}")
print(f"{'Query embedding':36}{ms(t_embed):>20}{'same':>20}")
print(f"{'FAISS Top 10 search':36}{ms(t_search):>20}{'same':>20}")
print(f"{'Cross-encoder scoring of 10 pairs':36}{'not run':>20}{ms(t_ce):>20}")
print(f"{'Total retrieval':36}{ms(tot1):>20}{ms(tot2):>20}")
print(f"{'LLM generation (separate, 3 calls)':36}{ms(t_llm):>20}{'(same prompt size)':>20}")
print(f"\nn = {len(t_embed)} timed runs ({len(timing_queries)} queries x {REPS}).  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

stage (ms/query, mean ± std)                  Pipeline 1          Pipeline 2
Query embedding                            99.2 ±    7.1                same
FAISS Top 10 search                         0.4 ±    0.2                same
Cross-encoder scoring of 10 pairs                not run      431.1 ±   22.1
Total retrieval                            99.6 ±    7.1      530.8 ±   23.2
LLM generation (separate, 3 calls)       1070.8 ±  154.4  (same prompt size)

n = 160 timed runs (8 queries x 20).  Device: CPU


## 9. Scaling check
Time the cross-encoder on 10, 50, 100 and 500 pairs, next to the FAISS search over the same corpus. The cross-encoder needs one forward pass per pair, so it should grow about linearly; flat FAISS search is a single matrix product over a small corpus.

In [13]:
q = byid["e1"]["query"]
qv = np.asarray(bi.encode([QUERY_PREFIX + q], normalize_embeddings=True), dtype="float32")
print(f"{'candidates':>10}{'cross-encoder ms':>20}{'FAISS search ms':>20}")
for n in (10, 50, 100, 500):
    _, ids = index.search(qv, n); ids = [int(i) for i in ids[0]]
    tc, tf = [], []
    for _ in range(5):
        t0 = time.perf_counter(); ce.predict([(q, chunks[i]["text"]) for i in ids], show_progress_bar=False); tc.append((time.perf_counter()-t0)*1e3)
        t0 = time.perf_counter(); index.search(qv, n); tf.append((time.perf_counter()-t0)*1e3)
    print(f"{n:>10}{np.mean(tc):>20.1f}{np.mean(tf):>20.2f}")
print(f"\nCorpus here: {index.ntotal} chunks. Scoring all of them with the cross-encoder is ~{index.ntotal} pairs per query.")

candidates    cross-encoder ms     FAISS search ms


        10               430.3                0.55


        50              1714.3                0.34


       100              3600.9                0.42


       500             17567.4                0.48

Corpus here: 1631 chunks. Scoring all of them with the cross-encoder is ~1631 pairs per query.



**Observed:** about 43, 34, 36 and 35 ms per pair at 10, 50, 100 and 500 candidates, so near-linear; the 10-pair figure includes fixed overhead. FAISS flat search stays at 0.3–0.6 ms over this range. Scoring all 1631 chunks would take roughly 57 s per query on this CPU (35 ms x 1631, extrapolated, not measured).


## 10. Chunk-size experiment (follow-up to section 3)
Does the 256-token choice matter? For 128, 256 and 512 tokens (overlap 15% of size, except 256 which uses the main run's 40 so its row matches section 7) the corpus is re-chunked and re-embedded, then for the same 40 queries I measure:
* **intact**: the gold phrase still appears whole in some chunk (if not, chunking made the fact unreachable);
* **recall@10**: gold in the dense Top 10;
* **Top 3 after re-rank**: gold in the re-ranked Top 3;
* **CE truncated**: chunks where query + chunk would exceed the cross-encoder's 512-token limit (for 512-token chunks essentially all of them).

One run, one corpus, 40 hand-written queries with substring-based gold: treat the numbers as an observation, not a general result.

In [14]:
def chunk_experiment(sizes=(128, 256, 512)):
    out = []
    for size in sizes:
        cks = build_chunks(pages, size, overlap=OVERLAP_TOKENS if size == CHUNK_TOKENS else round(size * 0.15))
        idx = build_index(cks)
        intact = r10 = top3 = 0
        for qid, typ, q, gold in CANDIDATES:
            if not any(_norm(gold) in _norm(c["text"]) for c in cks): continue
            intact += 1
            cands = dense_top_k(q, index=idx, k=K_RETRIEVE)
            if gold_rank(cands, gold, cks) is not None: r10 += 1
            ranked, _ = rerank(q, cands, cks)
            if (gold_rank(ranked, gold, cks) or 99) <= K_CONTEXT: top3 += 1
        n_trunc = sum(ntok(c["text"]) + 40 > 512 for c in cks)
        out.append((size, len(cks), intact, r10, top3, n_trunc))
        print(f"size {size:>3}: {len(cks):>5} chunks | intact {intact}/{len(CANDIDATES)} | recall@10 {r10} | re-ranked Top 3 {top3} | CE would truncate {n_trunc} chunks")
    return out

if RUN_CHUNK_EXPERIMENT:
    chunk_results = chunk_experiment()

size 128:  3257 chunks | intact 39/40 | recall@10 21 | re-ranked Top 3 17 | CE would truncate 0 chunks


size 256:  1631 chunks | intact 40/40 | recall@10 31 | re-ranked Top 3 23 | CE would truncate 0 chunks


size 512:   810 chunks | intact 40/40 | recall@10 30 | re-ranked Top 3 23 | CE would truncate 809 chunks


**Observed (single run):**
* 256 is best or tied on both columns 128 is worst and splits one phrase.
* 512 is not better than 256, and 809 of 810 chunks overflow the cross-encoder's 512 once ~40 query tokens are added (an estimate), so at 512 it scores truncated text.


## 11. Report 

### 11.1 Architecture comparison
* **Bi-encoder:** query and chunk go through the model separately, each giving one vector. Score = cosine / inner product. Chunk vectors are computed once, offline; a query costs one forward pass plus a dot-product search.
* **Cross-encoder:** query and chunk are joined into one input and self-attention runs across both, so each query word can attend to each chunk word (negations, "except", exact terms). Output is one relevance score per pair.
* **Consequence:** the cross-encoder score depends on the query, so nothing can be precomputed per chunk. It is accurate but needs one forward pass per pair, so it reorders a shortlist; the bi-encoder searches the whole corpus.

### 11.2 Rank shifts and answers
Per-query ranks and verdicts are in the table in 7.2.
* Gold in Top 3 (40 queries): baseline 24, re-ranked 23. Recovered 5, hurt 6, retrieval misses 9.
* Kept queries, gold rank base → re-ranked: k5 10→1, n2 9→1, b8 6→1, e1 10→2, e3 10→2, b5 1→1, n7 1→7, k6 miss.
* Answer 1 correct on 2 of 8 (b5, n7), Answer 2 on 4 of 8 (k5, n2, b8, b5). The rest were refusals ("Not found…"); no hallucination or misstated rule. When the key chunk sat outside the Top 3, the baseline omitted the answer rather than inventing one.

### 11.3 Latency (CPU, ms per query, 8 queries x 20 runs)
| stage | Pipeline 1 | Pipeline 2 |
|---|---|---|
| Query embedding | 99.2 ± 7.1 | same |
| FAISS Top 10 | 0.4 ± 0.2 | same |
| Cross-encoder, 10 pairs | not run | 431.1 ± 22.1 |
| **Total retrieval** | **99.6 ± 7.1** | **530.8 ± 23.2** |
| LLM generation (3 calls, separate) | 1070.8 ± 154.4 | same prompt size |

### 11.4 Scaling question
* A cross-encoder pass is needed for every (query, chunk) pair, and none can be precomputed. Cost grows linearly with corpus size.
* So the bi-encoder narrows the corpus to 10 cheaply and the cross-encoder spends its cost only on those. The limit: if the gold chunk is not in the 10 (9 of 40 here) re-ranking cannot recover it.

